# Thao tác Dữ liệu (CRUD) - Máy 1 (CN001)
Áp dụng tính **Phân tán** và **Trong suốt**

In [28]:
import pymongo
import datetime

IP_MAY_1 = '100.80.212.106'
IP_MAY_2 = '100.118.59.95'

db_name = 'fahasa_btl2'

print("Đang kết nối tới 2 cluster...")
client_1 = pymongo.MongoClient(f'mongodb://{IP_MAY_1}:27017/', serverSelectionTimeoutMS=5000)
client_2 = pymongo.MongoClient(f'mongodb://{IP_MAY_2}:27017/', serverSelectionTimeoutMS=5000)

# Định nghĩa sessions cho Máy 1: Local là client_1, Remote là client_2
clients = [client_1, client_2]

print("========== Kiểm tra kết nối 2 máy thuộc 2 cluster ==========")
print("Đã thiết lập kết nối thành công từ Máy 1 (CN1)!")

Đang kết nối tới 2 cluster...
========== Kiểm tra kết nối 2 máy thuộc 2 cluster ==========
Đã thiết lập kết nối thành công từ Máy 1 (CN1)!


In [29]:
def crud_ops(operation, clients, db_name, collection_name, data=None, condition=None):
    # Xử lý Insert
    if operation == 'insert':
        try:
            if 'MaCN' in data:
                # Logic trong suốt cho dữ liệu Phân mảnh (Fragmentation)
                target_client = clients[0] # Mặc định
                if data.get('MaCN') == 'CN001':
                    target_client = client_1
                elif data.get('MaCN') == 'CN002':
                    target_client = client_2
                
                target_client[db_name][collection_name].insert_one(data)
            else:
                # Logic trong suốt cho dữ liệu Nhân bản (Replication)
                for client in clients:
                    client[db_name][collection_name].insert_one(data)
            return True
        except Exception as e:
            print(f"Insert failed: {e}")
            return False

    # Xử lý Update / Delete
    success = False
    for i, client in enumerate(clients):
        try:
            count = client[db_name][collection_name].count_documents(condition)
            if count > 0:
                if operation == 'update':
                    client[db_name][collection_name].update_many(condition, data)
                elif operation == 'delete':
                    client[db_name][collection_name].delete_many(condition)
                
                success = True
            else:
                pass 
        except Exception as e:
            print(f"Error executing: {e}")
            continue
            
    if not success:
        print("Không tìm thấy dữ liệu ở bất kỳ chi nhánh nào trong hệ thống.")
        return False
        
    return True

## 1. Thêm dữ liệu
### Trường hợp demo thêm dữ liệu vào Máy 1 (Chi nhánh 1)
Máy 1 kiểm tra xem **Máy 2** đã thực hiện thêm nhân viên `NV9001` vào CN001 thành công hay chưa.

In [23]:
print("--- Máy 1 kiểm tra lại kết quả Insert của Máy 2 ---")
nv_cn1 = client_1[db_name]['nhanvien'].find_one({'_id': 'NV9001'})
print("Dữ liệu ở Máy 1 (CN1):", nv_cn1 if nv_cn1 else "Không tồn tại")

--- Máy 1 kiểm tra lại kết quả Insert của Máy 2 ---
Dữ liệu ở Máy 1 (CN1): {'_id': 'NV9001', 'MaCN': 'CN001', 'HoTen': 'Trần Lê Duy Tân', 'GioiTinh': 'Nam', 'NgaySinh': datetime.datetime(2005, 6, 16, 0, 0), 'SoDienThoai': '0999999991', 'NgayVaoLam': datetime.datetime(2026, 5, 29, 1, 4, 29, 444000)}


### Trường hợp demo thêm dữ liệu vào Máy 2 (Chi nhánh 2)
Máy 1 tiến hành thao tác thêm dữ liệu vào cơ sở dữ liệu ở CN002, sau đó kiểm tra tính đúng đắn (chỉ thêm vào CN002, không thêm nhầm vào CN001).

In [26]:
nv9002 = {
    '_id': 'NV9002', 'MaCN': 'CN002', 'HoTen': 'Lưu Quang Huy', 
    'GioiTinh': 'Nữ', 'NgaySinh': datetime.datetime(2005, 1, 4),
    'SoDienThoai': '0999999992', 'NgayVaoLam': datetime.datetime.now()
}

print("--- Máy 1 thực hiện Insert dữ liệu nhân viên NV9002 ---")
crud_ops('insert', clients, db_name, 'nhanvien', data=nv9002)

print("\n--- Máy 1 kiểm tra tính đúng đắn ở cả 2 chi nhánh ---")
nv_cn1 = client_1[db_name]['nhanvien'].find_one({'_id': 'NV9002'})
nv_cn2 = client_2[db_name]['nhanvien'].find_one({'_id': 'NV9002'})

print("\nDữ liệu ở Máy 1 (CN1):", nv_cn1 if nv_cn1 else "Không tồn tại")
print("\nDữ liệu ở Máy 2 (CN2):", nv_cn2 if nv_cn2 else "Không tồn tại")

--- Máy 1 thực hiện Insert dữ liệu nhân viên NV9002 ---

--- Máy 1 kiểm tra tính đúng đắn ở cả 2 chi nhánh ---

Dữ liệu ở Máy 1 (CN1): Không tồn tại

Dữ liệu ở Máy 2 (CN2): {'_id': 'NV9002', 'MaCN': 'CN002', 'HoTen': 'Lưu Quang Huy', 'GioiTinh': 'Nữ', 'NgaySinh': datetime.datetime(2005, 1, 4, 0, 0), 'SoDienThoai': '0999999992', 'NgayVaoLam': datetime.datetime(2026, 5, 29, 1, 9, 49, 831000)}


## 2. Cập nhật dữ liệu
**Tính trong suốt phân tán thể hiện trong trường hợp này như sau:** Khi Máy 1 thực hiện thao tác, logic xử lý sẽ tự động kiểm tra xem nhân viên đó được tạo ở chi nhánh nào. Từ đó hệ thống xác định được dữ liệu thuộc chi nhánh nào và tiến hành thao tác tương ứng trên chi nhánh đó (mà không cần người dùng/ứng dụng phải tự chỉ định cụ thể Máy 1 hay Máy 2).

### Trường hợp demo cập nhật dữ liệu ở Máy 1 (Chi nhánh 1)
Máy 1 kiểm tra xem **Máy 2** đã cập nhật số điện thoại của nhân viên `NV9001` thành công hay chưa.

In [30]:
print("--- Máy 1 kiểm tra kết quả Update của Máy 2 ---")
nv_cn1_upd = client_1[db_name]['nhanvien'].find_one({'_id': 'NV9001'})
print("Dữ liệu ở Máy 1 (CN1):", nv_cn1_upd if nv_cn1_upd else "Không tồn tại")

--- Máy 1 kiểm tra kết quả Update của Máy 2 ---
Dữ liệu ở Máy 1 (CN1): {'_id': 'NV9001', 'MaCN': 'CN001', 'HoTen': 'Trần Lê Duy Tân', 'GioiTinh': 'Nam', 'NgaySinh': datetime.datetime(2005, 6, 16, 0, 0), 'SoDienThoai': '0888888881', 'NgayVaoLam': datetime.datetime(2026, 5, 29, 1, 23, 30, 374000)}


### Trường hợp demo cập nhật dữ liệu ở Máy 2 (Chi nhánh 2)
**Máy 1** tiến hành cập nhật dữ liệu ở CN002, sau đó kiểm tra tính đúng đắn ở cả 2 chi nhánh.

In [31]:
print("--- Máy 1 thực hiện Update dữ liệu nhân viên NV9002 ---")
crud_ops('update', clients, db_name, 'nhanvien', condition={'_id': 'NV9002'}, data={'$set': {'SoDienThoai': '0888888882'}})

print("\n--- Máy 1 kiểm tra tính đúng đắn ở cả 2 chi nhánh ---")
nv_cn1_upd2 = client_1[db_name]['nhanvien'].find_one({'_id': 'NV9002'})
nv_cn2_upd2 = client_2[db_name]['nhanvien'].find_one({'_id': 'NV9002'})

print("\nDữ liệu ở Máy 1 (CN1):", nv_cn1_upd2 if nv_cn1_upd2 else "Không tồn tại")
print("\nDữ liệu ở Máy 2 (CN2):", nv_cn2_upd2 if nv_cn2_upd2 else "Không tồn tại")

--- Máy 1 thực hiện Update dữ liệu nhân viên NV9002 ---

--- Máy 1 kiểm tra tính đúng đắn ở cả 2 chi nhánh ---

Dữ liệu ở Máy 1 (CN1): Không tồn tại

Dữ liệu ở Máy 2 (CN2): {'_id': 'NV9002', 'MaCN': 'CN002', 'HoTen': 'Lưu Quang Huy', 'GioiTinh': 'Nữ', 'NgaySinh': datetime.datetime(2005, 1, 4, 0, 0), 'SoDienThoai': '0888888882', 'NgayVaoLam': datetime.datetime(2026, 5, 29, 1, 9, 49, 831000)}


## 3. Xóa dữ liệu
**Tính trong suốt phân tán thể hiện trong trường hợp này như sau:** Khi Máy 1 thực hiện thao tác, logic xử lý sẽ tự động kiểm tra xem nhân viên đó được tạo ở chi nhánh nào. Từ đó hệ thống xác định được dữ liệu thuộc chi nhánh nào và tiến hành thao tác tương ứng trên chi nhánh đó (mà không cần người dùng/ứng dụng phải tự chỉ định cụ thể Máy 1 hay Máy 2).

### Trường hợp demo xoá dữ liệu ở Máy 1 (Chi nhánh 1)
Máy 1 kiểm tra xem **Máy 2** đã xoá nhân viên `NV9001` thành công hay chưa.

In [32]:
print("--- Máy 1 kiểm tra kết quả Delete của Máy 2 ---")
nv_cn1_del = client_1[db_name]['nhanvien'].find_one({'_id': 'NV9001'})
print("Dữ liệu ở Máy 1 (CN1):", nv_cn1_del if nv_cn1_del else "Không tồn tại")

--- Máy 1 kiểm tra kết quả Delete của Máy 2 ---
Dữ liệu ở Máy 1 (CN1): Không tồn tại


### Trường hợp demo xoá dữ liệu ở Máy 2 (Chi nhánh 2)
**Máy 1** tiến hành xoá dữ liệu ở CN002, sau đó kiểm tra tính đúng đắn ở cả 2 chi nhánh.

In [33]:
print("--- Máy 1 thực hiện Delete dữ liệu nhân viên NV9002 ---")
crud_ops('delete', clients, db_name, 'nhanvien', condition={'_id': 'NV9002'})

print("\n--- Máy 1 kiểm tra tính đúng đắn ở cả 2 chi nhánh ---")
nv_cn1_del2 = client_1[db_name]['nhanvien'].find_one({'_id': 'NV9002'})
nv_cn2_del2 = client_2[db_name]['nhanvien'].find_one({'_id': 'NV9002'})

print("\nDữ liệu ở Máy 1 (CN1):", nv_cn1_del2 if nv_cn1_del2 else "Không tồn tại")
print("\nDữ liệu ở Máy 2 (CN2):", nv_cn2_del2 if nv_cn2_del2 else "Không tồn tại")

--- Máy 1 thực hiện Delete dữ liệu nhân viên NV9002 ---

--- Máy 1 kiểm tra tính đúng đắn ở cả 2 chi nhánh ---

Dữ liệu ở Máy 1 (CN1): Không tồn tại

Dữ liệu ở Máy 2 (CN2): Không tồn tại
